## Practical session 9: Dual approaches


Today's session is dedicated to algorithms exploiting duality. In fac,t constrained optimization problems admit a dual optimization problem, and one exploit this connection to design more efficient schemes.

In [ ]:
%load_ext autoreload
%autoreload 2

### Returning to an old inverse problem from signal processing


In Session 4, we considered a discretized signal reconstruction problem (from the paper *Nesterov's method in convex optimization*, N. J. Walkington, SIAM Review, 65(2), 2023). Let's do some recalls. The signal $u : [0,1] \rightarrow \mathbb{R}$  can be reconstructed from the noisy signal $\hat{u} : [0,1] \rightarrow \mathbb{R}$ by minimizing
$$
    f(u) = \int_0^1 \frac{1}{2} | u(t) - \hat{u}(t) |^2 dt + \frac{\alpha}{\beta} \int_0^1 | u' |^\beta dt,
$$
where $\alpha, \beta$ are parameters of the formulation. The regularization term in the above enforces smooth reconstructed signals. Suppose that we have $N$ points $\{\hat{x}_i\}_{i=0}^N$ such that $\hat{x}_i = \hat{u}(ih)$ with $h = 1 / N$ for every $i \in \{ 1,\dots,N\}$. One can reconstruct $u$ by constructing $\{x_i\}_{n=0}^N$ such that $x_i \approx u(i h)$ for every $i \in \{ 1,\dots,N\}$. This leads (using trapezoid approximations) to the following problem:
$$
    \min_{x \in \mathbb{R}^{N+1}} \frac{h}{4}\left( (x_0 - \hat{x}_0)^2 + (x_N - \hat{x}_N)^2 \right) + \frac{h}{2} \sum_{i=1}^N (x_i - \hat{x}_i)^2 + \frac{\alpha h^{1-\beta}}{\beta} \sum_{i=0}^{N-1} (x_i - x_{i+1})^\beta.
$$
In Session 4, we considered $\beta = 2$, in which case this problem is quadratic. Today, we will consider the case $\beta = 1$, which leads to a non-smooth optimization problem, whose resolution is not easily amenable to proximal gradient methods.

Let us rewrite the problem by introducing $p \in \mathbb{R}^N$ such that $p_i = x_{i+1} - x_i$. One can write this relation as $p = C u$ for some well-chosen matrix $C \in \mathbb{R}^{N \times N+1}$. We can then rewrite our problem as a constrained problem in the form
$$
\min_{Cx = p} f_1(x) + f_0(p)
$$
where $f_1(x) = \frac{1}{2} (x - \hat{x})^\top D (x - \hat{x})$ and $f_0(p) = \alpha \| p \|_1$. We can then introduce the Lagrangian associated to this problem as
$$
\mathcal{L}((x,p),\lambda) = f_1(x) + f_0(p) + \lambda^\top (p - Cx).
$$
We can then observe that 
$$
\min_{Cx = p} f_1(x) + f_0(p) = \min_{x,p} \max_\lambda \mathcal{L}((x,p),\lambda),
$$
leading to the dual problem
$$
\max_\lambda \min_{x,p} \mathcal{L}((x,p),\lambda),
$$
which can be shown to be equivalent to the constrained problem 
$$
\max_{\lambda \in [-\alpha,\alpha]^N} - \frac{1}{2} \lambda^\top (C D^{-1} C^\top) \lambda - (C \hat{x})^\top \lambda.
$$
Finally, we can rewrite this constrained maximization problem as a constrained minimization problem
$$
\min_{\lambda \in [-\alpha,\alpha]^N} \frac{1}{2} \lambda^\top (C D^{-1} C^\top) \lambda + (C \hat{x})^\top \lambda.
$$
Remark that this new problem is convex, with a quadratic objective, and with a constraint set admitting an easy-to-implement projection. But how is this useful for our original problem?

Suppose that $\lambda_*$ is solution to the dual problem and that $x_*$ is a solution to the primal problem. Then, $x_*$ satisfies the KKT conditions with multiplier $\lambda_*$. In particular, we can derive (by differentiating $\mathcal{L}$ with respect to $x$) that 
$$
x_* = \hat{x} + D^{-1} C^\top \lambda_*.
$$
Thus, finding $\lambda_*$ allows to derive $x_*$.

### Using a projected gradient descent algorithm to solve the dual problem

The dual problem ca be written as
$$
\min_{\lambda \in [-\alpha, \alpha]^N} g_1(\lambda),
$$
where $g_1(\lambda) = \frac{1}{2} \lambda^\top (C D^{-1} C^\top)\lambda + (C \hat{x})^\top \lambda$.
The file `problem.py` contains an implementation of $D$ and $C$, as well as the functions $f = f_1 + f_0$, $g_1$, and $\nabla g_1$.

> Compute the projection operator onto $[-\alpha, \alpha]^N$ and implement it in the file `problem.py`.

> Compute the Lipschitz constant of $\nabla g_1$ (one can exploit that $g_1$ is quadratic).

In [ ]:
### TO BE COMPLETED

> Run the projected gradient descent algorithm to solve the dual problem.

In [ ]:
from utils import *
import problem as pb
from algorithms import *

In [ ]:
### TO BE COMPLETED

> Use the function `from_lambda_to_x` that is implemented in the file `problem.py` to obtain the optimal $x_*$ from $\lambda_*$.

In [ ]:
### TO BE COMPLETED

> How do the reconstructed solutions look like for $\beta = 1$? Compare with the reconstructed solution for $\beta=2$ (recall that for $\beta=2$, the reconstruction problem is quadratic of the form $f(x) = \frac{1}{2}x^\top A x + b^\top x + c$, with $A$, $b$, and $c$, as well as $f$ and $\nabla f$ being given at the end of the file `problem.py`).

In [ ]:
### TO BE COMPLETED

In [ ]:
import matplotlib.pyplot as plt

plt.plot(np.linspace(0.0,1.0,pb.N+1),pb.x_hat, alpha=0.5, c='silver')
plt.plot(np.linspace(0.0,1.0,pb.N+1), [pb.u(t) for t in np.linspace(0.0,1.0,pb.N+1)],linewidth=2, c='k')
plt.plot(np.linspace(0.0,1.0,pb.N+1), x_star_cg, linewidth=2, label='CG')
plt.plot(np.linspace(0.0,1.0,pb.N+1), x_star_dual, linewidth=2, label='dual approach')
plt.legend()

### To go further: Influence of the parameters on the reconstruction problem

Just as in the case $\beta = 2$, the reconstruction also depends on the hyper-parameter $\alpha$.

> Perform the same analysis as before with $\alpha = 0.1$.

### To go further: Accelerated projected gradient descent

Projected gradient descent builds upon the gradient descent, for which we have seen as faster alternative: accelerated gradient descent. Suppose that we are trying to minimize $f$ over $C$, accelerated projected gradient descent starts from $x_0 = y_0$, and perform the following updates:
\begin{align*}
    x_{k+1} &= \textrm{proj}_C(y_k - \tau \nabla g(y_k))\\
    y_{k+1} &= x_{k+1} + \frac{\lambda_k - 1}{\lambda_{k+1}}(x_{k+1} - x_k).
\end{align*}

> Implement the above accelerated projected gradient descent algorithm and run it on the dual problem.